<a href="https://colab.research.google.com/github/zombimann/Mathematical-video-animations-and-visualization/blob/main/Unit_Disk_Blaschke_Quotient_Flow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Blaschke Quotient Flow Visualization

This notebook visualizes the dynamics of a **Blaschke Product Quotient** $R(z, t)$ within the unit disk $\mathbb{D} = \{z \in \mathbb{C} : |z| < 1\}$.

#### Mathematical Overview

1.  **Blaschke Factors**: The building blocks are automorphisms of the unit disk of the form:
    $$B_a(z) = \frac{z - a}{1 - \bar{a}z}$$
    where $a \in \mathbb{D}$. Each factor maps the unit circle to itself and has a single zero at $z=a$.

2.  **Quotient Flow**: We define a time-dependent rational function:
    $$R(z, t) = \frac{\prod_{j=1}^{N_z} B_{a_j(t)}(z)}{\prod_{k=1}^{N_p} B_{b_k(t)}(z)}$$
    The roots $a_j(t)$ and poles $b_k(t)$ follow continuous orbital paths within the disk.

#### Visualization Technique

*   **Domain Coloring**: The phase of $R(z, t)$ determines the hue, while the magnitude determines brightness and the presence of contour rings.
*   **Animation**: The roots (cyan dots) and poles (pink circles) move over time, with trails indicating their recent history.
*   **Boundary Stability**: By construction, the boundary $|z|=1$ remains invariant in magnitude (though not in phase), which is a key property of Blaschke products.

In [7]:
"""
Blaschke Quotient Flow  –  R(z,t) = ∏Bₐⱼ(t)(z) / ∏B_bₖ(t)(z)
inside the unit disk with domain colouring + animated roots/poles.
"""
import sys, os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib.patheffects as pe
from matplotlib.colors import hsv_to_rgb
from IPython.display import HTML

# ── settings ──────────────────────────────────────────────────────────────────
NX, NY   = 300, 300      # Slightly reduced for faster Colab rendering
N_FRAMES = 240          # 8 s @ 30 fps
FPS      = 30
DPI      = 100

BG         = "#07070f"
ZERO_COL   = "#00ffe0"
POLE_COL   = "#ff3d6b"
TRAIL_LEN  = 15

# ── maths ─────────────────────────────────────────────────────────────────────

def blaschke(z, a):
    return (z - a) / (1.0 - np.conj(a) * z)

def evaluate_R(Z, zeros, poles):
    val = np.ones_like(Z, dtype=complex)
    for a in zeros:
        val *= blaschke(Z, a)
    for b in poles:
        val /= blaschke(Z, b)
    return val

def domain_colour(w, inside):
    hue    = (np.angle(w) / (2 * np.pi)) % 1.0
    mag    = np.abs(w)
    logm   = np.log(np.maximum(mag, 1e-9))
    bright = np.exp(-0.45 * logm**2)
    rings  = 0.07 * np.sin(3 * logm)**2
    value  = np.clip(bright + rings, 0, 1)
    sat    = np.full_like(hue, 0.80)
    hsv    = np.stack([hue, sat, value], axis=-1)
    rgb    = hsv_to_rgb(hsv)
    alpha  = np.where(inside, 1.0, 0.0)[:, :, np.newaxis]
    return np.concatenate([rgb, alpha], axis=-1)

# ── animated paths ────────────────────────────────────────────────────────────

def orbit(r, fr, fth, pr, pth):
    def f(t):
        rho   = r * (0.55 + 0.45 * np.sin(fr  * 2*np.pi*t + pr))
        theta =       fth  * 2*np.pi*t + pth
        return rho * np.exp(1j * theta)
    return f

zero_orbs = [
    orbit(0.70, 0.18,  0.22,  0.0,  0.0),
    orbit(0.65, 0.13,  0.41,  1.1,  2.1),
    orbit(0.60, 0.09,  0.63,  2.5,  4.2),
]
pole_orbs = [
    orbit(0.72, 0.21, -0.17, 0.4,  1.0),
    orbit(0.58, 0.11, -0.38, 3.3,  3.5),
]
NZ, NP = len(zero_orbs), len(pole_orbs)

def positions(t):
    return (np.array([f(t) for f in zero_orbs]),
            np.array([f(t) for f in pole_orbs]))

# ── pixel grid ────────────────────────────────────────────────────────────────
x   = np.linspace(-1, 1, NX)
y   = np.linspace(-1, 1, NY)
X, Y = np.meshgrid(x, y)
Z    = X + 1j*Y
INSIDE = (X**2 + Y**2) < 1.0

# ── figure ────────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(8, 8.6), facecolor=BG)
ax  = fig.add_axes([0.0, 0.07, 1.0, 0.86])
ax.set_facecolor(BG)
ax.set_aspect("equal")
ax.set_xlim(-1.16, 1.16)
ax.set_ylim(-1.16, 1.16)
ax.axis("off")

# titles
glow = [pe.withStroke(linewidth=4, foreground=BG)]
fig.text(0.5, 0.978, "Blaschke Quotient Flow",
         ha="center", va="top", fontsize=21, fontweight="bold",
         color="#dde0ff", fontfamily="serif", path_effects=glow)
fig.text(0.5, 0.946,
         r"$R(z,t)=\dfrac{\prod B_{a_j(t)}(z)}{\prod B_{b_k(t)}(z)},\quad"
         r"B_a(z)=\dfrac{z-a}{1-\bar{a}z}$",
         ha="center", va="top", fontsize=12,
         color="#8888bb", fontfamily="serif")

# legend
for xp, col, lbl in [(0.33, ZERO_COL, "● zero  (root)"),
                      (0.67, POLE_COL, "○ pole")]:
    fig.text(xp, 0.028, lbl, ha="center", va="center",
             fontsize=11, color=col, fontfamily="monospace",
             path_effects=[pe.withStroke(linewidth=2, foreground=BG)])

# unit circle
th = np.linspace(0, 2*np.pi, 600)
ax.plot(np.cos(th), np.sin(th), color="white", lw=1.0, alpha=0.25, zorder=5)
ax.add_patch(plt.Circle((0,0), 1.0, color="#0d0d22", zorder=1))

# image object
img_obj = ax.imshow(np.zeros((NY, NX, 4)), extent=[-1,1,-1,1],
                     origin="lower", zorder=2, interpolation="bilinear")

# markers
def _mk(col, filled, sz, zord, alpha=1.0):
    fc = col if filled else "none"
    return ax.plot([], [], "o", color=fc, markeredgecolor=col,
                   markeredgewidth=2.2, markersize=sz,
                   zorder=zord, alpha=alpha)[0]

zmk = [_mk(ZERO_COL, True,  12, 11) for _ in range(NZ)]
pmk = [_mk(POLE_COL, False, 14, 11) for _ in range(NP)]
zhl = [_mk(ZERO_COL, True,  24, 4, alpha=0.07) for _ in range(NZ)]
phl = [_mk(POLE_COL, True,  26, 4, alpha=0.07) for _ in range(NP)]

ztr = [[_mk(ZERO_COL, True, max(2,10-k//4), 3,
             alpha=0.04+0.05*(TRAIL_LEN-k)/TRAIL_LEN) for k in range(TRAIL_LEN)]
       for _ in range(NZ)]
ptr = [[_mk(POLE_COL, False, max(2,11-k//4), 3,
             alpha=0.04+0.05*(TRAIL_LEN-k)/TRAIL_LEN) for k in range(TRAIL_LEN)]
       for _ in range(NP)]

ftxt = ax.text(1.12, -1.12, "", ha="right", va="bottom",
               fontsize=7, color="#33334a", fontfamily="monospace", zorder=20)

zh  = [[] for _ in range(NZ)]
ph  = [[] for _ in range(NP)]

# ── update ────────────────────────────────────────────────────────────────────

def update(frame):
    t  = frame / N_FRAMES
    zs, ps = positions(t)

    # domain colouring
    W    = np.where(INSIDE, evaluate_R(Z, zs, ps), 1+0j)
    rgba = domain_colour(W, INSIDE)
    img_obj.set_data(rgba)

    # trails
    for i,z in enumerate(zs):
        zh[i].append((z.real, z.imag))
        if len(zh[i]) > TRAIL_LEN: zh[i].pop(0)
    for i,p in enumerate(ps):
        ph[i].append((p.real, p.imag))
        if len(ph[i]) > TRAIL_LEN: ph[i].pop(0)

    for i in range(NZ):
        for k,tr in enumerate(ztr[i]):
            idx = len(zh[i])-1-k
            if idx >= 0: tr.set_data([zh[i][idx][0]], [zh[i][idx][1]])
            else:        tr.set_data([], [])
    for i in range(NP):
        for k,tr in enumerate(ptr[i]):
            idx = len(ph[i])-1-k
            if idx >= 0: tr.set_data([ph[i][idx][0]], [ph[i][idx][1]])
            else:        tr.set_data([], [])

    for i,z in enumerate(zs):
        zmk[i].set_data([z.real], [z.imag])
        zhl[i].set_data([z.real], [z.imag])
    for i,p in enumerate(ps):
        pmk[i].set_data([p.real], [p.imag])
        phl[i].set_data([p.real], [p.imag])

    ftxt.set_text(f"t = {t:.3f}")

    artists = [img_obj] + zmk + pmk + zhl + phl + [ftxt]
    for row in ztr: artists += row
    for row in ptr: artists += row
    return artists

# ── render ────────────────────────────────────────────────────────────────────
ani = animation.FuncAnimation(fig, update, frames=N_FRAMES, interval=1000//FPS, blit=True)
plt.close() # Prevents showing the last frame static

# Display in Colab
HTML(ani.to_html5_video())